In [1]:
import pandas as pd
import numpy as np
import holidays
import os
from pandas.api.types import CategoricalDtype
from transformers import pipeline
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

/Users/jyc/Desktop/DEV/projects/capstone_rta/rta-recommend/rta-venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/jyc/Desktop/DEV/projects/capstone_rta/rta-recommend/rta-venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 원본 데이터 불러오기

In [2]:
try:
    df = pd.read_csv('../data/raw/reviews.csv')
except FileNotFoundError:
    print("파일을 찾을 수 없습니다.")

print("### 로드 확인")
display(df.head())
print("\n### 데이터 정보 확인")
df.info()
print("\n### 결측치 확인")
print(df.isnull().sum)

### 로드 확인


,user_id,store_id,category,contents,time
0,139,10480,2,맛있게 잘 먹었습니다.,2025-07-12 18:05:00.000000
1,340,10480,2,사장님 저 감동이자나요.. 너무 친절하시고 감사합니다!! 치킨도 너무 맛있고 또 시...,2025-07-11 20:30:00.000000
2,46,10480,2,자극적인 매콤한 맛!,2025-06-22 19:20:00.000000
3,115,10480,2,아이가 잘먹고 염지가 순해서 좋아요.,2025-06-10 13:00:00.000000
4,253,10480,2,맛있게 잘 먹었습니다.,2025-06-07 18:30:00.000000



### 데이터 정보 확인
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3771 entries, 0 to 3770
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   user_id   3771 non-null   int64 
 1   store_id  3771 non-null   int64 
 2   category  3771 non-null   int64 
 3   contents  3771 non-null   object
 4   time      3771 non-null   object
dtypes: int64(3), object(2)
memory usage: 147.4+ KB

### 결측치 확인
<bound method DataFrame.sum of       user_id  store_id  category  contents   time
0       False     False     False     False  False
1       False     False     False     False  False
2       False     False     False     False  False
3       False     False     False     False  False
4       False     False     False     False  False
...       ...       ...       ...       ...    ...
3766    False     False     False     False  False
3767    False     False     False     False  False
3768    False     False     False     False  False
3769    

## time 칼럼 형식 변경 (object->datetime)

In [3]:
#6. time 칼럼 형식 변경 (object->datetime)
df['time'] = pd.to_datetime(df['time'], format="%Y-%m-%d %H:%M:%S.%f")

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3771 entries, 0 to 3770
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   user_id   3771 non-null   int64         
 1   store_id  3771 non-null   int64         
 2   category  3771 non-null   int64         
 3   contents  3771 non-null   object        
 4   time      3771 non-null   datetime64[ns]
dtypes: datetime64[ns](1), int64(3), object(1)
memory usage: 147.4+ KB


## time feature 세분화

In [4]:
# (1) meal_time
# (2) day_of_week
# (3) is_weekend
# (4) is_holiday

# --- meal_time ---
def get_meal_time(hour):
    if 5 <= hour < 11:
        return 'breakfast'
    elif 11 <= hour < 17:
        return 'lunch'
    elif 17 <= hour < 22:
        return 'dinner'
    else: # 22:00 ~ 04:59
        return 'late_night_snack'

# 'meal_time' 문자열 컬럼 생성
df['meal_time'] = df['time'].dt.hour.apply(get_meal_time)
# 'meal_time'을 정수 코드로 변환 (임베딩 레이어 입력용)
meal_time_order = ['breakfast', 'lunch', 'dinner', 'late_night_snack']
cat_type = CategoricalDtype(categories=meal_time_order, ordered=True)
df['meal_time_code'] = df['meal_time'].astype(cat_type).cat.codes


# --- day_of_week ---
# 요일 피처 생성 (0=월요일, 1=화요일, ..., 6=일요일)
df['day_of_week'] = df['time'].dt.dayofweek


# --- is_weekend---
# 주말 여부 피처 생성 (토요일=5 또는 일요일=6이면 1, 아니면 0)
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)


# --- is_holiday---
# 대한민국 공휴일 객체 생성
kr_holidays = holidays.KR()
# 공휴일 여부 피처 생성 (공휴일이면 1, 아니면 0)
df['is_holiday'] = df['time'].dt.date.apply(lambda x: x in kr_holidays).astype(int)


# --- 결과 확인 ---
final_features = ['time', 'meal_time_code', 'day_of_week', 'is_weekend', 'is_holiday']
display(df[final_features].head())

# meal_time 매핑 확인
meal_time_mapping = dict(enumerate(df['meal_time'].astype(cat_type).cat.categories))
print("### 'meal_time' 매핑 정보 ###")
print(meal_time_mapping)

,time,meal_time_code,day_of_week,is_weekend,is_holiday
0,2025-07-12 18:05:00,2,5,1,0
1,2025-07-11 20:30:00,2,4,0,0
2,2025-06-22 19:20:00,2,6,1,0
3,2025-06-10 13:00:00,1,1,0,0
4,2025-06-07 18:30:00,2,5,1,0


### 'meal_time' 매핑 정보 ###
{0: 'breakfast', 1: 'lunch', 2: 'dinner', 3: 'late_night_snack'}


## 감정 분석

In [5]:
# tqdm (진행 바)을 pandas apply와 연동
tqdm.pandas()

# 1. 감성 분석 파이프라인 로드
# GPU 사용 가능 시 device=0, CPU 사용 시 device=-1
sentiment_analyzer = pipeline(
    "sentiment-analysis", 
    model="daekeun-ml/koelectra-small-v3-nsmc",
    device=0
)

# 2. 분석 결과를 긍정(1)/부정(0)으로 변환하는 함수
def classify_sentiment(review_text):
    # 파이프라인은 [{'label': 'positive', 'score': 0.99}] 형태의 결과를 반환.
    result = sentiment_analyzer(review_text)
    if result[0]['label'] == '1':
        return 1  # 긍정
    else:
        return 0  # 부정

# 3. 'contents' 컬럼의 모든 리뷰에 함수를 적용하여 'is_positive' 컬럼 생성
df['is_positive'] = df['contents'].progress_apply(classify_sentiment)

# 라벨링 결과 확인
print("긍정/부정 라벨링 결과")
display(df.head(10))
print(f"\n긍정 리뷰 개수: {df['is_positive'].sum()}")
print(f"부정 리뷰 개수: {len(df) - df['is_positive'].sum()}")

Device set to use mps:0
100%|██████████| 3771/3771 [00:38<00:00, 97.25it/s] 

긍정/부정 라벨링 결과


,user_id,store_id,category,contents,time,meal_time,meal_time_code,day_of_week,is_weekend,is_holiday,is_positive
0,139,10480,2,맛있게 잘 먹었습니다.,2025-07-12 18:05:00,dinner,2,5,1,0,1
1,340,10480,2,사장님 저 감동이자나요.. 너무 친절하시고 감사합니다!! 치킨도 너무 맛있고 또 시...,2025-07-11 20:30:00,dinner,2,4,0,0,1
2,46,10480,2,자극적인 매콤한 맛!,2025-06-22 19:20:00,dinner,2,6,1,0,1
3,115,10480,2,아이가 잘먹고 염지가 순해서 좋아요.,2025-06-10 13:00:00,lunch,1,1,0,0,1
4,253,10480,2,맛있게 잘 먹었습니다.,2025-06-07 18:30:00,dinner,2,5,1,0,1
5,6,10480,2,먹다가 사진을 찍었네요. 아주 맛있네요~^^,2025-05-19 19:15:00,dinner,2,0,0,0,1
6,14,10480,2,사장님 무책임 40분 늦음. 주문 들어온지도 모름. 다 식은거 옴 사과 한마디도 없음,2025-05-16 22:05:00,late_night_snack,3,4,0,0,0
7,86,10480,2,처음 먹었는데 매콤한 감칠맛때문에 매우 맛있네요. ^^,2025-04-14 23:00:00,late_night_snack,3,0,0,0,1
8,197,10480,2,맛있게 잘 먹었습니다.,2025-03-30 21:45:00,dinner,2,6,1,0,1
9,175,10480,2,바삭바삭 고소하고 깔끔 맛있네요.,2025-03-28 19:35:00,dinner,2,4,0,0,1



긍정 리뷰 개수: 3471
부정 리뷰 개수: 300


## 데이터 분할

In [6]:
# 학습 및 테스트 데이터 분할 (8:2 비율)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

print(f"\n학습 데이터 개수: {len(train_df)}")
print(f"테스트 데이터 개수: {len(test_df)}")


# 전처리 완료된 데이터 파일로 저장
train_df.to_csv('../data/processed/train_data.csv', index=False)
test_df.to_csv('../data/processed/test_data.csv', index=False)

print("\n데이터 저장 완료.")


학습 데이터 개수: 3016
테스트 데이터 개수: 755

데이터 저장 완료.
